# Loan Default Prediction on Amazon SageMaker

This notebook walks through an end-to-end ML workflow for **loan default prediction** using Amazon SageMaker, covering:

1. **Synthetic data generation** — create a realistic 10M-row loan dataset with messy, real-world data quality issues
2. **Feature engineering** — run a distributed Ray Data pipeline on SageMaker Processing
3. **Model training** — train XGBoost and LightGBM models using SageMaker Training with Ray Train
4. **Hyperparameter tuning** — use SageMaker Automatic Model Tuning (AMT) to find optimal hyperparameters
5. **SageMaker Pipelines** — stitch everything into a repeatable, parameterized MLOps pipeline

Each section includes the launch code and a brief explanation of the SageMaker SDK concepts involved.

**Prerequisites:** This notebook assumes you are running in a SageMaker environment (Studio, notebook instance, or local with valid credentials) with an IAM role that has SageMaker and S3 permissions.

This notebook works with the SageMaker SDK v2. We must pin SageMaker SDK to v2

In [ ]:
!pip install sagemaker==2.245.0

SageMaker Default Configuration

To avoid repeating common parameters (role, tags, VPC, encryption) across training and processing jobs, you can define them once in a config file.

Edit sagemaker-default-config/config.yaml with your defaults, then point the SDK to it before creating a Session:
```python
import os
os.environ["SAGEMAKER_USER_CONFIG_OVERRIDE"] = f"{os.getcwd()}/sagemaker-default-config"
```
>   Note: The environment variable should point to the directory containing config.yaml, not the file itself.


## Setup

⚠️ Configure S3 paths and SageMaker session in the code block below. Update these to match your environment.

In [6]:
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
default_bucket = session.default_bucket()

# ── S3 paths (update these for your environment) ──
S3_BUCKET = "mmfsi"
S3_PREFIX = "loan-fe-v2"

RAW_DATA_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/raw_loan_data.csv"
FE_OUTPUT_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/features"
MODEL_OUTPUT_S3 = f"s3://{default_bucket}/loan-models/"

# After FE, training data lands here
TRAIN_S3 = f"{FE_OUTPUT_S3}/train"
VAL_S3 = f"{FE_OUTPUT_S3}/val"

print(f"Role:    {role}")
print(f"Region:  {region}")
print(f"Bucket:  {default_bucket}")

sagemaker.config INFO - Fetched defaults config from location: /home/sagemaker-user/OMF/data-processing/sagemaker-default-config/config.yaml
Role:    arn:aws:iam::692323308711:role/trainBlog-ExecutionRole-6MHIIJEX9Z2M
Region:  us-east-1
Bucket:  sagemaker-us-east-1-692323308711


---
## 1. Generate Synthetic Loan Data

The `generate_loan_data.py` script creates a realistic loan dataset with intentional data quality issues that mirror real-world data:

- **Missing values** — MCAR, MAR, and MNAR patterns (3-8% per column)
- **String inconsistencies** — mixed casing, typos, synonyms (e.g. `"Married"`, `"MARRIED"`, `"Maried"`)
- **Numeric outliers** — impossible ages, credit scores outside valid range
- **Mixed date formats** — `YYYY-MM-DD`, `MM/DD/YYYY`, `DD-MM-YYYY`, `DD Mon YYYY`, plus garbage values
- **Duplicate rows** — ~1% injected per chunk
- **Imbalanced target** — ~15% Default, ~77% Paid, ~8% Current

The generator is optimized for scale (100M+ rows) using vectorized NumPy operations and multiprocessing.

We generate **10M rows** locally and then upload to S3 for the SageMaker processing job.

In [2]:
!python generate_loan_data.py --rows 10000000 --format csv --output raw_loan_data.csv

Generating 10,000,000 rows in 20 chunks of up to 500,000
Workers: 8 | Format: CSV
Output:  raw_loan_data.csv

  Chunk 1/20:  505,000 rows in 8.7s | Total:      505,000 / 10,000,000 | 54,182 rows/s
  Chunk 2/20:  505,000 rows in 8.8s | Total:    1,010,000 / 10,000,000 | 108,136 rows/s
  Chunk 3/20:  505,000 rows in 8.8s | Total:    1,515,000 / 10,000,000 | 162,041 rows/s
  Chunk 4/20:  505,000 rows in 8.9s | Total:    2,020,000 / 10,000,000 | 214,457 rows/s
  Chunk 5/20:  505,000 rows in 8.8s | Total:    2,525,000 / 10,000,000 | 268,019 rows/s
  Chunk 6/20:  505,000 rows in 8.9s | Total:    3,030,000 / 10,000,000 | 320,357 rows/s
  Chunk 7/20:  505,000 rows in 9.0s | Total:    3,535,000 / 10,000,000 | 371,733 rows/s
  Chunk 8/20:  505,000 rows in 9.3s | Total:    4,040,000 / 10,000,000 | 410,169 rows/s
  Chunk 9/20:  505,000 rows in 8.4s | Total:    4,545,000 / 10,000,000 | 255,163 rows/s
  Chunk 10/20:  505,000 rows in 8.4s | Total:    5,050,000 / 10,000,000 | 282,848 rows/s
  Chunk 11

In [3]:
# Upload raw data to S3 for the SageMaker Processing job
import boto3

s3 = boto3.client("s3")
local_file = "raw_loan_data.csv"
s3_key = f"{S3_PREFIX}/raw_loan_data.csv"

print(f"Uploading {local_file} to s3://{S3_BUCKET}/{s3_key} ...")
s3.upload_file(local_file, S3_BUCKET, s3_key)
print(f"Upload complete: {RAW_DATA_S3}")

Uploading raw_loan_data.csv to s3://mmfsi/loan-fe-v2/raw_loan_data.csv ...
Upload complete: s3://mmfsi/loan-fe-v2/raw_loan_data.csv


---
## 2. Feature Engineering with SageMaker Processing

### How SageMaker Processing works

[SageMaker Processing](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html) runs your data processing code on managed infrastructure. Key concepts:

- **Processor** — defines the container image, instance type, and instance count. We use `PyTorchProcessor` here — not because we use PyTorch for FE, but because it provides a well-maintained container with Python/pip support where we install Ray via `requirements.txt`.
- **ProcessingInput** — maps an S3 path to a local path *inside the container*. SageMaker downloads the data before your script starts.
  - `source`: S3 URI of the input data
  - `destination`: container-local path (e.g. `/opt/ml/processing/input/data`)
  - `s3_data_distribution_type`: `"FullyReplicated"` (default, all nodes get full data) or `"ShardedByS3Key"` (each node gets a subset)
- **ProcessingOutput** — maps a container-local path back to S3. SageMaker uploads it when the job finishes.
  - `source`: container-local output path (e.g. `/opt/ml/processing/output/data`)
  - `destination`: S3 URI where results land
- **source_dir** — local directory containing your entry point script and `requirements.txt`. SageMaker tars and uploads it, then installs dependencies before running.

**Container path layout:**
```
/opt/ml/processing/
├── input/data/          ← ProcessingInput downloads data here
│   └── raw_loan_data.csv
└── output/data/         ← Your script writes results here → ProcessingOutput uploads to S3
    ├── train/           ← Training split (partitioned CSVs)
    └── val/             ← Validation split
```

Our FE script (`src/sagemaker_run.py`) bootstraps a multi-node Ray cluster across all SageMaker instances, then runs a 9-stage Ray Data pipeline that cleans, normalizes, engineers 41 features, and produces a train/val split.

> **Docs:** [PyTorchProcessor](https://sagemaker.readthedocs.io/en/stable/frameworks/pytorch/sagemaker.pytorch.html#pytorch-processor) | [Processing I/O](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html#processing-job-container)

In [ ]:
from sagemaker.pytorch import PyTorchProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

processor = PyTorchProcessor(
    role=role,
    instance_count=2,                   # 2-node Ray cluster for distributed FE
    instance_type="ml.m5.4xlarge",       # 16 vCPU, 64 GB — good for data processing
    framework_version="2.5.1",           # PyTorch container version (hosts Ray via requirements.txt)
    py_version="py311",
    sagemaker_session=session,
    base_job_name="loan-feature-engineering",
    volume_size_in_gb=30                  # size of attached storage
)

processor.run(
    code="sagemaker_run.py",             # Entry point — bootstraps Ray cluster, then runs FE pipeline
    source_dir="src",                    # Contains: sagemaker_run.py, feature_engineering.py, requirements.txt
    inputs=[
        ProcessingInput(
            source=RAW_DATA_S3,                              # S3 URI of the raw CSV
            destination="/opt/ml/processing/input/data",     # Where SageMaker puts it inside the container
            s3_data_distribution_type="FullyReplicated",     # Every node gets the full file
        ),
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output/data",         # Where FE script writes train/val splits
            destination=FE_OUTPUT_S3,                        # S3 destination for the output
        ),
    ],
    arguments=[
        "--input", "/opt/ml/processing/input/data/raw_loan_data.csv",
        "--output", "/opt/ml/processing/output/data",
        "--format", "csv",
        "--batch-size", "65536",
        "--val-split", "0.2"            # 80/20 train/val split
    ],
    wait=True,                          # Set to True to block until complete
    logs=True,
)

sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.EnableNetworkIsolation
sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.EnableInterContainerTrafficEncryption
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.EnableInterContainerTrafficEncryption
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.EnableInterContainerTrafficEncryption
sagemaker.config INFO - Applied value(s) from config key = SageMaker.ProcessingJob.Tags


INFO:sagemaker:Creating processing-job with name loan-feature-engineering-2026-04-14-23-52-51-151


.......

#### list the feature engineering artifacts

In [ ]:
!aws s3 ls {FE_OUTPUT_S3}/ --recursive

---
## 3. Model Training with SageMaker Estimators

### How SageMaker Training works

[SageMaker Training](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-training.html) runs your training code on managed compute, handles data download/upload, and emits metrics to CloudWatch. Key concepts:

- **Estimator** — wraps your training script with infrastructure config. We use the `PyTorch` estimator (same reason as Processing — we need a Python container with pip to install Ray, XGBoost/LightGBM).
  - `entry_point`: the script SageMaker invokes (e.g. `sagemaker_train.py`)
  - `source_dir`: directory with your code + `requirements.txt`
  - `hyperparameters`: passed to your script as CLI arguments (`--learning-rate 0.1`)
  - `metric_definitions`: regex patterns to extract metrics from stdout into CloudWatch
- **Input channels** — named data sources passed to `.fit()`. SageMaker downloads them to:
  - `training` channel → `/opt/ml/input/data/training/`
  - `validation` channel → `/opt/ml/input/data/validation/`
- **Model artifacts** — your script saves the model to `/opt/ml/model/`. SageMaker tars and uploads it to `output_path` in S3.

**Container path layout:**
```
/opt/ml/
├── input/
│   └── data/
│       ├── training/      ← "training" channel data (partitioned CSVs)
│       └── validation/    ← "validation" channel data
├── model/                 ← Your script saves model artifacts here → uploaded to S3
└── output/                ← Failure details written here
```

Both training scripts use Ray Train under the hood. For multi-node runs, `sagemaker_train.py` bootstraps a Ray cluster across SageMaker instances (similar to the FE job), then uses `DataParallelTrainer` to distribute gradient computation.

> **Docs:** [SageMaker Training](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-training.html) | [PyTorch Estimator](https://sagemaker.readthedocs.io/en/stable/frameworks/pytorch/sagemaker.pytorch.html#pytorch-estimator)

### 3a. Distributed XGBoost Training

Uses `src_dist_training/` which contains:
- `sagemaker_train.py` — SageMaker entry point (Ray cluster bootstrap + training dispatch)
- `train_xgboost_dist.py` — core XGBoost training logic using Ray Train's `DataParallelTrainer`
- `requirements.txt` — installs `ray[data,train]`, `xgboost`, `scikit-learn`

**Metric definitions** tell SageMaker how to parse metrics from stdout using regex. These appear in the CloudWatch training job metrics and are required for HPO to optimize against.

In [128]:
from sagemaker.pytorch import PyTorch

# Regex building block for a floating-point number
_NUM = r"([-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?)"

# CloudWatch metric definitions for XGBoost — covers binary, multiclass, regression
XGB_METRIC_DEFINITIONS = [
    # Final evaluation metrics (printed by training script)
    {"Name": "auc-roc",    "Regex": rf"AUC-ROC:\s+{_NUM}"},
    {"Name": "auc-pr",     "Regex": rf"AUC-PR:\s+{_NUM}"},
    {"Name": "f1",         "Regex": rf"F1:\s+{_NUM}"},
    {"Name": "accuracy",   "Regex": rf"Accuracy:\s+{_NUM}"},
    {"Name": "f1-macro",   "Regex": rf"F1 \(macro\):\s+{_NUM}"},
    {"Name": "f1-weighted", "Regex": rf"F1 \(weight\):\s+{_NUM}"},
    {"Name": "rmse",       "Regex": rf"RMSE:\s+{_NUM}"},
    {"Name": "mae",        "Regex": rf"MAE:\s+{_NUM}"},
    {"Name": "r2",         "Regex": rf"R..?:\s+{_NUM}"},
    # XGBoost per-round metrics (from xgb.train verbose output)
    {"Name": "xgb:eval-auc",      "Regex": rf"eval-auc:{_NUM}"},
    {"Name": "xgb:eval-logloss",  "Regex": rf"eval-logloss:{_NUM}"},
    {"Name": "xgb:eval-mlogloss", "Regex": rf"eval-mlogloss:{_NUM}"},
    {"Name": "xgb:eval-rmse",     "Regex": rf"eval-rmse:{_NUM}"},
    {"Name": "xgb:eval-error",    "Regex": rf"eval-error:{_NUM}"},
    {"Name": "xgb:eval-merror",   "Regex": rf"eval-merror:{_NUM}"},
]

xgb_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir="src_dist_training",
    role=role,
    instance_count=1,                    # Single node multiple workers; increase for multi-node Ray
    instance_type="ml.m5.4xlarge",       # CPU, for large dataset GPU is more efficient
    framework_version="2.5.1",
    py_version="py311",
    output_path=MODEL_OUTPUT_S3,
    sagemaker_session=session,
    base_job_name="loan-xgb-dist-training",
    hyperparameters={
        "objective": "binary:logistic",
        "tree-method": "hist",
        "num-workers": 0,                # 0 = auto-detect from cluster topology
        "num-rounds": 500,
        "early-stopping": 20,
        "max-depth": 7,
        "learning-rate": 0.1,
        "subsample": 0.8,
        "colsample-bytree": 0.8,
        "min-child-weight": 5
    },
    metric_definitions=XGB_METRIC_DEFINITIONS,
    keep_alive_period_in_seconds=1800,       # Keep Instance warm for quick iteration, https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html
    max_run=7200,                             # maximum amount of time the job can run for, caps at 28 days.
    disable_profiler=True,
    disable_output_compression=True
)

xgb_estimator.fit(
    {"training": TRAIN_S3, "validation": VAL_S3},
    wait=False,                               # Unblock the jupyter notebook cell so you can execute other cells
    logs=True,
)

xgb_job = xgb_estimator.latest_training_job.name
print(f"XGBoost job submitted: {xgb_job}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: loan-xgb-dist-training-2026-04-14-21-16-59-919


XGBoost job submitted: loan-xgb-dist-training-2026-04-14-21-16-59-919


### 3.b To see the logs of the jobs programmatically if you set `wait=False`, you can run the command below

In [ ]:
my_estimator = sagemaker.estimator.Estimator.attach(xgb_job)
# View logs
my_estimator.logs()

### 3.c Download Model artifact an test locally

In [ ]:
# download model from s3 to loacal dir `model` using aws cli commands
!aws s3 sync {xgb_estimator.model_data['S3DataSource']['S3Uri']} model/xgb/

In [149]:
# Content of extract artifact
!ls model/xgb/

feature_importance.json  metrics.json  model.json  model.ubj  params.json


In [150]:
import xgboost as xgb
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

model = xgb.Booster()
model.load_model("model/xgb/model.json")

# Load test data (first column = labels)
df = pd.read_csv("test.csv")

# Split labels and features
y_true = df["target"]
X_test = df.drop(columns=["target"])

# Create DMatrix and predict
dtest = xgb.DMatrix(X_test, label=y_true)
y_prob = model.predict(dtest)        # probabilities
y_pred = (y_prob > 0.5).astype(int)  # binary predictions

# Evaluate
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_true, y_prob):.4f}")
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["Paid", "Default"]))

Accuracy:  0.5911
ROC AUC:   0.6869

Classification Report:
              precision    recall  f1-score   support

        Paid       0.92      0.57      0.71     15969
     Default       0.21      0.70      0.32      2595

    accuracy                           0.59     18564
   macro avg       0.57      0.64      0.51     18564
weighted avg       0.82      0.59      0.65     18564



### 3b. Distributed LightGBM Training

Same pattern as XGBoost but uses `src_dist_training_lgbm/`. LightGBM uses `tree_learner="data_parallel"` for distributed gradient synchronization (vs. XGBoost's Rabit AllReduce). Note the different hyperparameter names: `num-leaves` instead of `min-child-weight`, and a different objective string (`binary` vs `binary:logistic`).

In [ ]:
LGBM_METRIC_DEFINITIONS = [
    # Final evaluation metrics (printed by training script)
    {"Name": "auc-roc",    "Regex": rf"AUC-ROC:\s+{_NUM}"},
    {"Name": "auc-pr",     "Regex": rf"AUC-PR:\s+{_NUM}"},
    {"Name": "f1",         "Regex": rf"F1:\s+{_NUM}"},
    {"Name": "accuracy",   "Regex": rf"Accuracy:\s+{_NUM}"},
    {"Name": "f1-macro",   "Regex": rf"F1 \(macro\):\s+{_NUM}"},
    {"Name": "f1-weighted", "Regex": rf"F1 \(weight\):\s+{_NUM}"},
    {"Name": "rmse",       "Regex": rf"RMSE:\s+{_NUM}"},
    {"Name": "mae",        "Regex": rf"MAE:\s+{_NUM}"},
    {"Name": "r2",         "Regex": rf"R2:\s+{_NUM}"},
    # LightGBM per-round metrics (from lgb.log_evaluation callback)
    {"Name": "lgbm:eval-auc",         "Regex": rf"eval's auc: {_NUM}"},
    {"Name": "lgbm:eval-logloss",     "Regex": rf"eval's binary_logloss: {_NUM}"},
    {"Name": "lgbm:eval-mlogloss",    "Regex": rf"eval's multi_logloss: {_NUM}"},
    {"Name": "lgbm:eval-rmse",        "Regex": rf"eval's rmse: {_NUM}"},
    {"Name": "lgbm:eval-l2",          "Regex": rf"eval's l2: {_NUM}"},
    {"Name": "lgbm:eval-multi-error", "Regex": rf"eval's multi_error: {_NUM}"},
]

lgbm_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir="src_dist_training_lgbm",   # LightGBM-specific training code
    role=role,
    instance_count=2,
    instance_type="ml.m5.4xlarge",         # current LightGBM implementation does not support GPU, only use CPU
    framework_version="2.5.1",
    py_version="py311",
    output_path=MODEL_OUTPUT_S3,
    sagemaker_session=session,
    base_job_name="loan-lgbm-dist-training",
    hyperparameters={
        "objective": "binary",               # LightGBM uses "binary", not "binary:logistic"
        "num-workers": 0,
        "num-rounds": 500,
        "early-stopping": 20,
        "max-depth": 7,
        "num-leaves": 127,                   # LightGBM-specific: controls tree complexity
        "learning-rate": 0.1,
        "subsample": 0.8,
        "colsample-bytree": 0.8,
        "min-child-samples": 20             # LightGBM equivalent of min-child-weight
    },
    metric_definitions=LGBM_METRIC_DEFINITIONS,
    keep_alive_period_in_seconds=1800,       # keep instance warm for faster start up with same job config, max of an hour
    max_run=7200,                             # maximum amount of time the job can run for, caps at 28 days.
    disable_profiler=True,
    disable_output_compression=True
)

lgbm_estimator.fit(
    {"training": TRAIN_S3, "validation": VAL_S3},
    wait=True,
    logs=True,
)

In [ ]:
# download model from s3 to loacal dir `model` using aws cli commands
!aws s3 sync {lgbm_estimator.model_data['S3DataSource']['S3Uri']} model/lgbm/

In [146]:
# list contents of downloaded artifact
!ls model/lgbm/

feature_importance.json  metrics.json  model.txt  params.json


In [147]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Load model
model = lgb.Booster(model_file="model/lgbm/model.txt")

# Load test data
df = pd.read_csv("test.csv")

# Split labels and features — keep as DataFrame
y_true = df["target"].values
X_test = df.drop(columns=["target"])

# Predict (LightGBM returns probabilities directly for binary)
y_prob = model.predict(X_test)
y_pred = (y_prob > 0.5).astype(int)

# Evaluate
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_true, y_prob):.4f}")
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["Paid", "Default"]))

Accuracy:  0.8602
ROC AUC:   0.6849

Classification Report:
              precision    recall  f1-score   support

        Paid       0.86      1.00      0.92     15969
     Default       0.00      0.00      0.00      2595

    accuracy                           0.86     18564
   macro avg       0.43      0.50      0.46     18564
weighted avg       0.74      0.86      0.80     18564



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


---
## 4. Hyperparameter Tuning with SageMaker AMT

### How SageMaker Automatic Model Tuning works

[SageMaker AMT](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html) (Automatic Model Tuning) runs multiple training jobs with different hyperparameter combinations to find the best-performing model. Key concepts:

- **`HyperparameterTuner`** — orchestrates the search. Configure:
  - `objective_metric_name`: the metric to optimize (must match a name in `metric_definitions`)
  - `objective_type`: `"Maximize"` (e.g. AUC) or `"Minimize"` (e.g. RMSE)
  - `strategy`: `"Bayesian"` (uses prior results to guide search — more efficient) or `"Random"`
  - `max_jobs`: total number of training jobs to launch
  - `max_parallel_jobs`: how many run concurrently (higher = faster but less Bayesian benefit)
- **Hyperparameter ranges** — define the search space:
  - `ContinuousParameter(min, max)` — for floats (e.g. learning rate)
  - `IntegerParameter(min, max)` — for integers (e.g. max depth)
  - `CategoricalParameter(["a", "b"])` — for discrete choices
- **Multi-algorithm tuning** — `HyperparameterTuner.create()` accepts `estimator_dict` and `hyperparameter_ranges_dict` to search over multiple algorithms simultaneously. The tuner allocates jobs across algorithms and ranks all trials by the objective metric.

**Static vs. tuned hyperparameters:** HPs set on the estimator's `hyperparameters` dict are *static* (fixed for every trial). HPs in `hyperparameter_ranges` are *tuned* (the tuner varies them). Don't put the same HP in both.

> **Docs:** [AMT Overview](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html) | [HyperparameterTuner API](https://sagemaker.readthedocs.io/en/stable/api/training/tuner.html)

### 4a. XGBoost Hyperparameter Tuning

We create a fresh estimator with only *static* hyperparameters (objective, tree-method, num-rounds, early-stopping) and let the tuner vary the rest.

In [157]:
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter

# Estimator with only STATIC hyperparameters (tuned ones go in hp_ranges)
xgb_tune_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir="src_dist_training",
    role=role,
    instance_count=1,
    instance_type="ml.m5.4xlarge",
    framework_version="2.5.1",
    py_version="py311",
    output_path=MODEL_OUTPUT_S3,
    sagemaker_session=session,
    base_job_name="loan-hpo-xgboost",
    hyperparameters={
        "objective": "binary:logistic",
        "tree-method": "hist",
        "num-rounds": 500,
        "early-stopping": 20,
        "num-workers": 0
    },
    metric_definitions=XGB_METRIC_DEFINITIONS,
    disable_profiler=True,
    disable_output_compression=True
)

# Define search space — the tuner will sample from these ranges
xgb_hp_ranges = {
    "learning-rate":    ContinuousParameter(0.01, 0.3),
    "max-depth":        IntegerParameter(3, 10),
    "subsample":        ContinuousParameter(0.5, 1.0),
    "colsample-bytree": ContinuousParameter(0.5, 1.0),
    "min-child-weight": IntegerParameter(1, 10),
}

xgb_tuner = HyperparameterTuner(
    estimator=xgb_tune_estimator,
    metric_definitions=XGB_METRIC_DEFINITIONS,
    objective_metric_name="auc-roc",         # Must match a Name in metric_definitions
    objective_type="Maximize",
    hyperparameter_ranges=xgb_hp_ranges,
    strategy="Bayesian",                     # Uses GP surrogate to pick promising HPs
    max_jobs=8,                             # Total training jobs
    max_parallel_jobs=4,                     # Concurrent jobs (lower = better Bayesian guidance)
    base_tuning_job_name="loan-hpo-xgboost",
)

xgb_tuner.fit(
    {"training": TRAIN_S3, "validation": VAL_S3},
    wait=False
)

xgb_tuning_job = xgb_tuner.latest_tuning_job.name
print(f"XGBoost HPO job: {xgb_tuning_job}")
print(f"Monitor: https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/hyper-tuning-jobs/{xgb_tuning_job}")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating hyperparameter tuning job with name: loan-hpo-xgboost-260414-2126


XGBoost HPO job: loan-hpo-xgboost-260414-2126
Monitor: https://us-east-1.console.aws.amazon.com/sagemaker/home?region=us-east-1#/hyper-tuning-jobs/loan-hpo-xgboost-260414-2126


### To view the tuning job progress programmatically, run the below command. You may have to wait a few seconds for the metadata to populate

In [169]:
xgb_tuner.analytics().dataframe()

,colsample-bytree,learning-rate,max-depth,min-child-weight,subsample,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
0,0.721690,0.052732,8.0,7.0,0.504739,loan-hpo-xgboost-260414-2126-008-2c61ad37,Completed,0.680422,2026-04-14 21:33:29+00:00,2026-04-14 21:36:49+00:00,200.0
1,0.937461,0.042107,8.0,3.0,0.999941,loan-hpo-xgboost-260414-2126-007-96a50a3c,Completed,0.680984,2026-04-14 21:32:10+00:00,2026-04-14 21:35:04+00:00,174.0
2,0.687747,0.116183,4.0,4.0,0.991661,loan-hpo-xgboost-260414-2126-006-2eb6183d,Completed,0.681042,2026-04-14 21:31:58+00:00,2026-04-14 21:35:07+00:00,189.0
3,0.600627,0.176351,9.0,6.0,0.611695,loan-hpo-xgboost-260414-2126-005-f07399a9,Completed,0.680380,2026-04-14 21:31:37+00:00,2026-04-14 21:33:51+00:00,134.0
4,0.741337,0.061403,4.0,6.0,0.508037,loan-hpo-xgboost-260414-2126-004-87505ebc,Completed,0.681028,2026-04-14 21:27:48+00:00,2026-04-14 21:33:04+00:00,316.0
5,0.690047,0.154102,4.0,6.0,0.990468,loan-hpo-xgboost-260414-2126-003-f056ac07,Completed,0.680998,2026-04-14 21:27:46+00:00,2026-04-14 21:31:41+00:00,235.0
6,0.550892,0.224856,7.0,5.0,0.605196,loan-hpo-xgboost-260414-2126-002-2e04ab65,Completed,0.680687,2026-04-14 21:27:49+00:00,2026-04-14 21:31:12+00:00,203.0
7,0.510187,0.262698,3.0,8.0,0.563673,loan-hpo-xgboost-260414-2126-001-81c6635f,Completed,0.680698,2026-04-14 21:27:43+00:00,2026-04-14 21:31:52+00:00,249.0


#### Get best tuner candidate and test the model locally

In [164]:
# get best training candidate
best_job = xgb_tuner.best_training_job()
print(f"Best job: {best_job}")

# load best candidate
best_estimator = sagemaker.estimator.Estimator.attach(best_job)

# download the artifacts locally, by default tuner does not inherit the `disable_output_compression=True` and compresses the output, so we will need to uncompress
!aws s3 cp {best_estimator.model_data} model/tuner-xgb/

# untar model output
import tarfile
with tarfile.open("model/tuner-xgb/model.tar.gz", "r:gz") as tar:
    tar.extractall(path="./model/tuner-xgb")

Best job: loan-hpo-xgboost-260414-2126-006-2eb6183d

2026-04-14 21:35:10 Starting - Found matching resource for reuse
2026-04-14 21:35:10 Downloading - Downloading the training image
2026-04-14 21:35:10 Training - Training image download completed. Training in progress.
2026-04-14 21:35:10 Uploading - Uploading generated training model
2026-04-14 21:35:10 Completed - Resource released due to keep alive period expiry
download: s3://sagemaker-us-east-1-692323308711/loan-models/loan-hpo-xgboost-260414-2126-006-2eb6183d/output/model.tar.gz to model/tuner-xgb/model.tar.gz


/tmp/ipykernel_425581/4215158210.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="./model/tuner-xgb")


In [167]:
model = xgb.Booster()
model.load_model("model/tuner-xgb/model.json")

# Load test data (first column = labels)
df = pd.read_csv("test.csv")

# Split labels and features
y_true = df["target"]
X_test = df.drop(columns=["target"])

# Create DMatrix and predict
dtest = xgb.DMatrix(X_test, label=y_true)
y_prob = model.predict(dtest)        # probabilities
y_pred = (y_prob > 0.5).astype(int)  # binary predictions

# Evaluate
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_true, y_prob):.4f}")
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["Paid", "Default"]))

Accuracy:  0.5893
ROC AUC:   0.6868

Classification Report:
              precision    recall  f1-score   support

        Paid       0.92      0.57      0.71     15969
     Default       0.21      0.70      0.32      2595

    accuracy                           0.59     18564
   macro avg       0.57      0.64      0.51     18564
weighted avg       0.82      0.59      0.65     18564



### 4b. LightGBM Hyperparameter Tuning

LightGBM has 6 tunable hyperparameters (one more than XGBoost: `num-leaves`, which controls the leaf count directly rather than relying solely on depth).

In [139]:
lgbm_tune_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir="src_dist_training_lgbm",
    role=role,
    instance_count=1,
    instance_type="ml.m5.4xlarge",
    framework_version="2.5.1",
    py_version="py311",
    output_path=MODEL_OUTPUT_S3,
    sagemaker_session=session,
    base_job_name="loan-hpo-lightgbm",
    hyperparameters={
        "objective": "binary",
        "num-rounds": 500,
        "early-stopping": 20,
        "num-workers": 0
    },
    metric_definitions=LGBM_METRIC_DEFINITIONS,
    disable_profiler=True,
    disable_output_compression=True
)

lgbm_hp_ranges = {
    "learning-rate":     ContinuousParameter(0.01, 0.3),
    "max-depth":         IntegerParameter(3, 10),
    "num-leaves":        IntegerParameter(15, 255),     # LightGBM leaf-wise growth control
    "subsample":         ContinuousParameter(0.5, 1.0),
    "colsample-bytree":  ContinuousParameter(0.5, 1.0),
    "min-child-samples": IntegerParameter(5, 50),
}

lgbm_tuner = HyperparameterTuner(
    estimator=lgbm_tune_estimator,
    metric_definitions=LGBM_METRIC_DEFINITIONS,
    objective_metric_name="auc-roc",
    objective_type="Maximize",
    hyperparameter_ranges=lgbm_hp_ranges,
    strategy="Bayesian",
    max_jobs=20,
    max_parallel_jobs=4,
    base_tuning_job_name="loan-hpo-lightgbm",
)

lgbm_tuner.fit(
    {"training": TRAIN_S3, "validation": VAL_S3},
    wait=False,
    logs=False,
)

lgbm_tuning_job = lgbm_tuner.latest_tuning_job.name
print(f"LightGBM HPO job: {lgbm_tuning_job}")
print(f"Monitor: https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/hyper-tuning-jobs/{lgbm_tuning_job}")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating hyperparameter tuning job with name: loan-hpo-lightgbm-260414-2121


LightGBM HPO job: loan-hpo-lightgbm-260414-2121
Monitor: https://us-east-1.console.aws.amazon.com/sagemaker/home?region=us-east-1#/hyper-tuning-jobs/loan-hpo-lightgbm-260414-2121


In [168]:
lgbm_tuner.analytics().dataframe()

,colsample-bytree,learning-rate,max-depth,min-child-samples,num-leaves,subsample,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
0,0.743630,0.126408,6.0,11.0,176.0,0.852294,loan-hpo-lightgbm-260414-2121-020-2025ca75,Completed,0.678693,2026-04-14 21:32:48+00:00,2026-04-14 21:34:58+00:00,130.0
1,0.733852,0.089841,6.0,31.0,251.0,0.573770,loan-hpo-lightgbm-260414-2121-019-e9a11f35,Completed,0.678183,2026-04-14 21:32:47+00:00,2026-04-14 21:34:47+00:00,120.0
2,0.730202,0.023357,6.0,36.0,163.0,0.704720,loan-hpo-lightgbm-260414-2121-018-08e625ee,Completed,0.679234,2026-04-14 21:32:47+00:00,2026-04-14 21:34:51+00:00,124.0
3,0.781462,0.221755,6.0,47.0,218.0,0.606291,loan-hpo-lightgbm-260414-2121-017-d4cd56a0,Completed,0.678308,2026-04-14 21:32:21+00:00,2026-04-14 21:34:21+00:00,120.0
4,0.776000,0.155205,6.0,5.0,173.0,0.539627,loan-hpo-lightgbm-260414-2121-016-693984ad,Completed,0.678611,2026-04-14 21:30:26+00:00,2026-04-14 21:32:25+00:00,119.0
5,0.767962,0.251467,6.0,11.0,86.0,0.786355,loan-hpo-lightgbm-260414-2121-015-3acdf107,Completed,0.678632,2026-04-14 21:30:25+00:00,2026-04-14 21:32:24+00:00,119.0
6,0.778710,0.152596,6.0,29.0,192.0,0.771186,loan-hpo-lightgbm-260414-2121-014-55ffa033,Completed,0.678566,2026-04-14 21:30:26+00:00,2026-04-14 21:32:24+00:00,118.0
7,0.997815,0.183201,6.0,38.0,239.0,0.989184,loan-hpo-lightgbm-260414-2121-013-1b0927b6,Completed,0.678697,2026-04-14 21:29:59+00:00,2026-04-14 21:31:58+00:00,119.0
8,0.942274,0.126240,6.0,13.0,248.0,0.758892,loan-hpo-lightgbm-260414-2121-012-1e6c53af,Completed,0.678549,2026-04-14 21:27:59+00:00,2026-04-14 21:29:58+00:00,119.0
9,0.878513,0.206369,6.0,29.0,248.0,0.969624,loan-hpo-lightgbm-260414-2121-011-73c32ccf,Completed,0.678081,2026-04-14 21:27:56+00:00,2026-04-14 21:29:56+00:00,120.0


### 4c. Multi-Algorithm Hyperparameter Tuning

SageMaker allows you to tune multiple algorithms simultaneously, searching for the best algorithm and hyperparameter combination in a single job. Rather than running separate tuning jobs and comparing results manually, the tuner explores the hyperparameter space for each algorithm independently and ranks all trials by a shared objective metric.

Here we launch a tuning job that pits XGBoost against LightGBM, letting SageMaker determine which algorithm and parameter configuration yields the best performance.

In [171]:
# Infrastructure
INSTANCE_TYPE = "ml.m5.4xlarge"
WAIT = False

# HPO strategy
MAX_JOBS = 20
MAX_PARALLEL_JOBS = 4
STRATEGY = "Bayesian"          # "Bayesian", "Random", "Grid" or "HyperBand"
OBJECTIVE_METRIC = "auc-roc"
OBJECTIVE_TYPE = "Maximize"    # "Maximize" or "Minimize"

# Shared training settings
NUM_ROUNDS = 500
EARLY_STOPPING = 20

# Per-algorithm objectives
XGB_OBJECTIVE = "binary:logistic"
XGB_NUM_CLASS = None            # set for multiclass (e.g. 5)
LGBM_OBJECTIVE = "binary"
LGBM_NUM_CLASS = None           # set for multiclass (e.g. 5)

# ── XGBoost estimator ───────────────────────────────────────────────────────

xgb_hyperparameters = {
    "objective": XGB_OBJECTIVE,
    "tree-method": "hist",
    "num-rounds": NUM_ROUNDS,
    "early-stopping": EARLY_STOPPING,
    "num-workers": 0            # System autodetect
}
if XGB_NUM_CLASS is not None:
    xgb_hyperparameters["num-class"] = XGB_NUM_CLASS

xgb_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir="src_dist_training",
    role=role,
    instance_count=1,
    instance_type=INSTANCE_TYPE,
    framework_version="2.5.1",
    py_version="py311",
    output_path=MODEL_OUTPUT_S3,
    sagemaker_session=session,
    base_job_name="loan-hpo-xgboost",
    hyperparameters=xgb_hyperparameters,
    metric_definitions=XGB_METRIC_DEFINITIONS,
    disable_profiler=True,
    disable_output_compression=True
)

xgb_hp_ranges = {
    "learning-rate":     ContinuousParameter(0.01, 0.3),
    "max-depth":         IntegerParameter(3, 10),
    "subsample":         ContinuousParameter(0.5, 1.0),
    "colsample-bytree":  ContinuousParameter(0.5, 1.0),
    "min-child-weight":  IntegerParameter(1, 10),
}

# ── LightGBM estimator ──────────────────────────────────────────────────────

lgbm_hyperparameters = {
    "objective": LGBM_OBJECTIVE,
    "num-rounds": NUM_ROUNDS,
    "early-stopping": EARLY_STOPPING,
    "num-workers": 0
}
if LGBM_NUM_CLASS is not None:
    lgbm_hyperparameters["num-class"] = LGBM_NUM_CLASS

lgbm_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir="src_dist_training_lgbm",
    role=role,
    instance_count=1,
    instance_type=INSTANCE_TYPE,
    framework_version="2.5.1",
    py_version="py311",
    output_path=MODEL_OUTPUT_S3,
    sagemaker_session=session,
    base_job_name="loan-hpo-lightgbm",
    hyperparameters=lgbm_hyperparameters,
    metric_definitions=LGBM_METRIC_DEFINITIONS,
    disable_profiler=True,
    disable_output_compression=True
)

lgbm_hp_ranges = {
    "learning-rate":      ContinuousParameter(0.01, 0.3),
    "max-depth":          IntegerParameter(3, 10),
    "num-leaves":         IntegerParameter(15, 255),
    "subsample":          ContinuousParameter(0.5, 1.0),
    "colsample-bytree":   ContinuousParameter(0.5, 1.0),
    "min-child-samples":  IntegerParameter(5, 50),
}

tuner = HyperparameterTuner.create(
    estimator_dict={
        "xgboost": xgb_estimator,
        "lightgbm": lgbm_estimator,
    },
    objective_metric_name_dict={
        "xgboost": OBJECTIVE_METRIC,
        "lightgbm": OBJECTIVE_METRIC,
    },
    hyperparameter_ranges_dict={
        "xgboost": xgb_hp_ranges,
        "lightgbm": lgbm_hp_ranges,
    },
    metric_definitions_dict={
        "xgboost": XGB_METRIC_DEFINITIONS,
        "lightgbm": LGBM_METRIC_DEFINITIONS,
    },
    objective_type=OBJECTIVE_TYPE,
    strategy=STRATEGY,
    max_jobs=MAX_JOBS,
    max_parallel_jobs=MAX_PARALLEL_JOBS,
    base_tuning_job_name="loan-hpo-xgb-lgbm",
)

# ── Input channels (same data for both algorithms) ──────────────────────────

channels = {"training": TRAIN_S3, "validation": VAL_S3}

inputs = {
    "xgboost": channels,
    "lightgbm": channels,
}

# ── Launch ───────────────────────────────────────────────────────────────────

print("Launching HPO job...")
tuner.fit(
    inputs=inputs,
    include_cls_metadata={},
    wait=WAIT,
    logs=WAIT,
)

tuning_job_name = tuner.latest_tuning_job.name
if WAIT:
    print(f"\nHPO job complete: {tuning_job_name}")
else:
    print(f"\nHPO job submitted: {tuning_job_name}")

print(
    f"Monitor at: https://{session.boto_region_name}.console.aws.amazon.com/"
    f"sagemaker/home?region={session.boto_region_name}"
    f"#/hyper-tuning-jobs/{tuning_job_name}"
)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating hyperparameter tuning job with name: loan-hpo-xgb-lgbm-260414-2143


Launching HPO job...

HPO job submitted: loan-hpo-xgb-lgbm-260414-2143
Monitor at: https://us-east-1.console.aws.amazon.com/sagemaker/home?region=us-east-1#/hyper-tuning-jobs/loan-hpo-xgb-lgbm-260414-2143


### Check Status

In [170]:
tuner.analytics().dataframe()

,colsample-bytree,learning-rate,max-depth,min-child-weight,subsample,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds,TrainingJobDefinitionName,min-child-samples,num-leaves
0,0.993502,0.010395,6.0,2.0,0.543200,loan-hpo-xgb-lgbm-260414-1855-020-581fe85d,Completed,0.680958,2026-04-14 19:11:59+00:00,2026-04-14 19:28:10+00:00,971.0,xgboost,NaN,NaN
1,0.993552,0.036921,6.0,6.0,0.870396,loan-hpo-xgb-lgbm-260414-1855-019-6d9b26c5,Completed,0.680950,2026-04-14 19:11:48+00:00,2026-04-14 19:18:58+00:00,430.0,xgboost,NaN,NaN
2,0.981968,0.035850,6.0,7.0,0.723800,loan-hpo-xgb-lgbm-260414-1855-018-95c7960a,Completed,0.680949,2026-04-14 19:11:24+00:00,2026-04-14 19:18:17+00:00,413.0,xgboost,NaN,NaN
3,0.522395,0.035913,8.0,5.0,0.912227,loan-hpo-xgb-lgbm-260414-1855-017-c029a33c,Completed,0.680175,2026-04-14 19:10:41+00:00,2026-04-14 19:13:51+00:00,190.0,xgboost,NaN,NaN
4,0.876581,0.040692,7.0,9.0,0.818784,loan-hpo-xgb-lgbm-260414-1855-016-5fe74134,Completed,0.680809,2026-04-14 19:08:07+00:00,2026-04-14 19:11:37+00:00,210.0,xgboost,NaN,NaN
5,0.873713,0.034988,7.0,7.0,0.628565,loan-hpo-xgb-lgbm-260414-1855-015-3f77bfa3,Completed,0.680820,2026-04-14 19:08:03+00:00,2026-04-14 19:11:08+00:00,185.0,xgboost,NaN,NaN
6,0.959391,0.036817,5.0,3.0,0.809731,loan-hpo-xgb-lgbm-260414-1855-014-d1e50fd6,Completed,0.680848,2026-04-14 19:06:25+00:00,2026-04-14 19:10:20+00:00,235.0,xgboost,NaN,NaN
7,0.810807,0.021022,6.0,3.0,0.621075,loan-hpo-xgb-lgbm-260414-1855-013-188fea10,Completed,0.680843,2026-04-14 19:05:03+00:00,2026-04-14 19:11:28+00:00,385.0,xgboost,NaN,NaN
8,0.751340,0.094304,7.0,2.0,0.969753,loan-hpo-xgb-lgbm-260414-1855-012-61534321,Completed,0.680731,2026-04-14 19:04:26+00:00,2026-04-14 19:07:10+00:00,164.0,xgboost,NaN,NaN
9,0.976533,0.015020,10.0,1.0,0.919449,loan-hpo-xgb-lgbm-260414-1855-011-4f43f4bd,Completed,0.680575,2026-04-14 19:03:57+00:00,2026-04-14 19:07:37+00:00,220.0,xgboost,NaN,NaN


---
## 5. SageMaker Pipelines — MLOps Template

### Why SageMaker Pipelines?

The steps above (FE, training, tuning) work well for ad-hoc experimentation, but for a repeatable MLOps workflow you want:

- **Reproducibility** — a versioned DAG definition with all parameters captured
- **Parameterization** — override S3 paths, instance types, hyperparameters, or algorithm choice at runtime without editing code
- **Conditional logic** — choose XGBoost, LightGBM, or both; toggle between training and HPO
- **Step dependencies** — FE must complete before training starts; its output feeds the training input channels
- **Auditability** — every execution is tracked in the SageMaker console with full lineage

### Pipeline architecture

We have two pipeline scripts:

1. **`pipeline_fe_training.py`** — Full pipeline: Feature Engineering → Training/Tuning
2. **`pipeline_training.py`** — Training/Tuning only (assumes FE data already exists in S3)

Both are config-driven via `pipeline_config.yaml`. Settings fall into two categories:
- **Baked at creation time** — `source_dir`, `entry_point`, `tuning_ranges`, `framework_version`. Changes require re-running the pipeline script.
- **Overridable at execution time** — everything in `parameters:` block. Override via `--params Key=Value` or the SageMaker console.

### Pipeline graph (FE → Training)

```
FeatureEngineering (ProcessingStep)
    │
    ├── CheckXgbTraining     → TrainXGBoost     (if algo=xgboost|both AND tuning=false)
    ├── CheckLgbmTraining    → TrainLightGBM    (if algo=lightgbm|both AND tuning=false)
    ├── CheckXgbTuning       → TuneXGBoost      (if algo=xgboost|both AND tuning=true)
    ├── CheckLgbmTuning      → TuneLightGBM     (if algo=lightgbm|both AND tuning=true)
    └── CheckBothAlgoTuning  → TuneBothAlgo     (if algo=both AND tuning=true)
```

### Key SageMaker Pipeline SDK concepts

| Concept | Description |
|---------|-------------|
| **`Pipeline`** | The top-level DAG. Contains parameters, steps, and session. |
| **`PipelineSession`** | A deferred session — `.fit()` and `.run()` don't execute immediately; they return step arguments. |
| **`ParameterString` / `ParameterInteger`** | Typed parameters that can be overridden at execution time. |
| **`ProcessingStep`** | Wraps a Processor `.run()` call. |
| **`TrainingStep`** | Wraps an Estimator `.fit()` call. |
| **`TuningStep`** | Wraps a `HyperparameterTuner.fit()` call. |
| **`ConditionStep`** | Branch logic — evaluates conditions and runs `if_steps` or `else_steps`. |
| **`ConditionIn` / `ConditionEquals`** | Condition primitives: check if a parameter is in a list or equals a value. |
| **`Join`** | String concatenation for building S3 paths from parameters (e.g. `Join(on="/", values=[fe_output_s3, "train"])`). |
| **`pipeline.upsert()`** | Creates or updates the pipeline definition. Idempotent. |
| **`pipeline.start()`** | Kicks off an execution. Pass `parameters={"Key": "value"}` to override defaults. |

> **Docs:** [SageMaker Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html) | [Pipeline SDK](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/index.html) | [Condition Steps](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps.html#step-type-condition)

Here is the DAG flow of the pipeline, full pipeline including feature engineering. The goal here is to create a template that can be reused, more specifically for training, the training only pipeline is `pipeline_training.py`, so users dont have to write scripts everytime they need to train a model.


<img src="images/pipeline-sm.png" width="1000"/>

### View the pipeline script

The cell below syntax-highlights the full FE-to-Training pipeline script so you can read through the pipeline construction inline.

In [ ]:
!pygmentize pipeline_fe_training.py

### Launch the pipeline

Before running the cell below, review `pipeline_config.yaml` and update:
- **S3 paths** (`FeRawInputS3Uri`, `FeOutputS3Uri`, `TrainS3Uri`, `ValS3Uri`, `OutputS3Uri`) to match your bucket
- **Instance types** if you want different compute
- **`role`** if you need a specific IAM role (set to `null` for auto-detect)

You can override any parameter at execution time without modifying the config file.

In [180]:
import importlib
import pipeline_fe_training

# After making changes to my_module.py
importlib.reload(pipeline_fe_training)


<module 'pipeline_fe_training' from '/home/sagemaker-user/OMF/data-processing/pipeline_fe_training.py'>

In [182]:
import yaml
from pipeline_fe_training import create_pipeline, load_config

config = load_config("pipeline_config.yaml")

# Build the pipeline
pipeline = create_pipeline(config, role)

# Create / update the pipeline definition in SageMaker
pipeline.upsert(role_arn=role)
pipeline_name = config["pipeline"].get("fe_training_name", "loan-fe-training-pipeline")
print(f"Pipeline ready: {pipeline_name}")

# Start an execution — override parameters as needed
execution = pipeline.start(
    parameters={
        "AlgorithmChoice": "both",       # "xgboost", "lightgbm", or "both"
        "RunTuning": "true",            # "true" to run HPO instead of single training
        "FeRawInputS3Uri": RAW_DATA_S3,  # Point to the data we generated
        "FeOutputS3Uri": FE_OUTPUT_S3,
        "OutputS3Uri": MODEL_OUTPUT_S3
    }
)

# print(f"\nExecution ARN: {execution.arn}")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.processing:Uplo

Pipeline ready: loan-fe-training-sm-pipeline-1


---
## Summary

| Step | SageMaker SDK | What it does |
|------|--------------|--------------|
| Data generation | Local script | 10M synthetic loan rows with realistic data quality issues |
| Feature engineering | `PyTorchProcessor` + `ProcessingStep` | Distributed Ray Data pipeline: cleaning, normalization, 41 engineered features, train/val split |
| XGBoost training | `PyTorch` estimator + `TrainingStep` | Ray Train distributed XGBoost with early stopping |
| LightGBM training | `PyTorch` estimator + `TrainingStep` | Ray Train distributed LightGBM with data-parallel tree learning |
| Hyperparameter tuning | `HyperparameterTuner` + `TuningStep` | Bayesian search over 5-6 HPs per algorithm, optimizing AUC-ROC |
| Pipeline | `Pipeline` with `ConditionStep` | Config-driven DAG: FE → conditional training/tuning for one or both algorithms |

### Next steps

- **Add a model evaluation step** — use a `ProcessingStep` to compute metrics on a held-out test set and gate deployment with a `ConditionStep`
- **Register the best model** — add a `ModelStep` or `RegisterModel` step to push to SageMaker Model Registry
- **Deploy** — create an endpoint with `model.deploy()` or add an endpoint step to the pipeline
- **CI/CD** — trigger `pipeline.start()` from a CodePipeline / GitHub Actions workflow on every merge to main